In [7]:
import pandas as pd
bank = pd.read_csv("C:\\Users\\tassili\\Documents\\E-comerce Conversion Funnel and Campaign Analytics\\01_raw_data\\bank-additional-full.csv", sep=';')
print(bank.shape)
print(bank.columns.tolist())
print(bank.dtypes)
bank.head

(41188, 21)
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object


<bound method NDFrame.head of        age          job  marital            education  default housing loan  \
0       56    housemaid  married             basic.4y       no      no   no   
1       57     services  married          high.school  unknown      no   no   
2       37     services  married          high.school       no     yes   no   
3       40       admin.  married             basic.6y       no      no   no   
4       56     services  married          high.school       no      no  yes   
...    ...          ...      ...                  ...      ...     ...  ...   
41183   73      retired  married  professional.course       no     yes   no   
41184   46  blue-collar  married  professional.course       no      no   no   
41185   56      retired  married    university.degree       no     yes   no   
41186   44   technician  married  professional.course       no      no   no   
41187   74      retired  married  professional.course       no     yes   no   

         contact mont

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

print(bank['contact'].value_counts())
print(bank['y'].value_counts())

print("===2===")
  
bank_clean = bank[bank['contact'].isin(['cellular', 'telephone'])].copy()  # a new data frame with only cellular and telephone contacts because we want to compare these two groups we add copy 
print(bank_clean.shape)

print("===3===")
conversion_summary = bank_clean.groupby('contact')['y'].apply(
    lambda x : pd.Series({
        'total': len(x),
        'subscribed': (x == 'yes').sum(),
        'conversion_rate': round((x == 'yes').mean()*100, 2)
    })
).unstack()
print(conversion_summary)

print("===4===")
contingency = pd.crosstab(bank_clean['contact'], bank_clean['y']) #jdwl t9at3i for chi square test
print(contingency)

print("===5===")
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"chi2_statistic: {chi2:.4f}")
print(f"p-value:{p_value:.10f}")
print(f"degrees of freedom: {dof}")
print("Expected frequencies:\n", expected)

print("===6===")
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(f"Cramer's V (Effect Size): {cramers_v:.4f}")

print("===7===")

rate_cellular = conversion_summary.loc['cellular', 'conversion_rate']
rate_telephone = conversion_summary.loc['telephone', 'conversion_rate']
absolute_difference = abs(rate_cellular - rate_telephone)
relative_lift = (absolute_difference / rate_telephone) * 100  
print(f"Absolute difference in conversion rates: {absolute_difference:.2f}%")
print(f"relative lift: {relative_lift:.2f}%")


conversion_summary.to_csv(r"C:\Users\tassili\Documents\E-comerce Conversion Funnel and Campaign Analytics\04_power_bi\conversion_summary.csv")

contact
cellular     26144
telephone    15044
Name: count, dtype: int64
y
no     36548
yes     4640
Name: count, dtype: int64
===2===
(41188, 21)
===3===
             total  subscribed  conversion_rate
contact                                        
cellular   26144.0      3853.0            14.74
telephone  15044.0       787.0             5.23
===4===
y             no   yes
contact               
cellular   22291  3853
telephone  14257   787
===5===
chi2_statistic: 862.3184
p-value:0.0000000000
degrees of freedom: 1
Expected frequencies:
 [[23198.7693503  2945.2306497]
 [13349.2306497  1694.7693503]]
===6===
Cramer's V (Effect Size): 0.1447
===7===
Absolute difference in conversion rates: 9.51%
relative lift: 181.84%


In [9]:
from statsmodels.stats.proportion import confint_proportions_2indep

# ============ 1) Confidence Interval ============
n_cellular, n_telephone = 26144, 15044
x_cellular, x_telephone = 3853, 787

ci_diff = confint_proportions_2indep(
    x_cellular, n_cellular, x_telephone, n_telephone,
    method='wald'
)
print(f"95% CI for difference in conversion rates: {ci_diff[0]*100:.2f}% to {ci_diff[1]*100:.2f}%")

# ============ 2) فحص جودة كامل لـ bank ============
print("\n=== Nulls ===")
print(bank.isnull().sum())

print("\n=== Duplicates ===")
print(bank.duplicated().sum())

print("\n=== Outliers (age) ===")
print(bank['age'].describe())
print("Age > 90:", (bank['age'] > 90).sum())

print("\n=== Outliers (duration, للتوثيق فقط) ===")
print(bank['duration'].describe())
print("Duration = 0 (لم تتم المكالمة فعليًا):", (bank['duration'] == 0).sum())

print("\n=== Unknown values check ===")
for col in bank.select_dtypes(include='object').columns:
    unknown_count = (bank[col] == 'unknown').sum()
    if unknown_count > 0:
        print(f"{col}: {unknown_count} unknown values")

# ============ 3) ملاحظة توثيقية عن duration ============
duration_note = """
Note: The 'duration' column (last contact duration in seconds) was intentionally 
EXCLUDED from this analysis. It is only known after a call ends, meaning it cannot 
be used for realistic pre-call decision-making, and it strongly correlates with the 
outcome (longer calls almost always end in 'yes'). Including it would leak information 
about the outcome. Only 'contact' method was used as the comparison variable.
"""
print(duration_note)

95% CI for difference in conversion rates: 8.95% to 10.06%

=== Nulls ===
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

=== Duplicates ===
12

=== Outliers (age) ===
count    41188.00000
mean        40.02406
std         10.42125
min         17.00000
25%         32.00000
50%         38.00000
75%         47.00000
max         98.00000
Name: age, dtype: float64
Age > 90: 10

=== Outliers (duration, للتوثيق فقط) ===
count    41188.000000
mean       258.285010
std        259.279249
min          0.000000
25%        102.000000
50%        180.000000
75%        319.000000
max       4918.000000
Name: duration, dtype

In [10]:
export_summary = conversion_summary.reset_index()
export_summary['ci_lower'] = ci_diff[0] * 100
export_summary['ci_upper'] = ci_diff[1] * 100
export_summary['chi2_pvalue'] = p_value
export_summary['cramers_v'] = cramers_v

export_summary.to_csv('C:\\Users\\tassili\\Documents\\E-comerce Conversion Funnel and Campaign Analytics\\04_power_bi\\campaign_comparison.csv', index=False)
print("Saved successfully.")
print(export_summary)

Saved successfully.
     contact    total  subscribed  conversion_rate  ci_lower   ci_upper  \
0   cellular  26144.0      3853.0            14.74   8.94841  10.064162   
1  telephone  15044.0       787.0             5.23   8.94841  10.064162   

     chi2_pvalue  cramers_v  
0  1.525986e-189   0.144693  
1  1.525986e-189   0.144693  
